## Running the Control group City - Quebec City

**By running the Difference-in-Difference analysis, we need one control group city, this city should have following features:**

1. **Similar to the treatment group city**: The control city should have similar characteristics to the treatment city, such as population size, economic structure, and demographic profile.

2. **Not affected by the treatment**: The control city should not be directly affected by the policy changes being studied, ensuring that any differences observed can be attributed to the treatment rather than other factors.

3. **Stable over time**: The control city should have a stable trend in the outcome variable of interest before the treatment, allowing for a clear comparison with the treatment city.

4. **Geographical proximity**: Ideally, the control city should be geographically close to the treatment city to minimize external influences and ensure that both cities are subject to similar regional economic conditions, including the peak time and seasonality of the year to the STR market.

Here, we use the Vancouver city as the control group city, which is similar to the treatment group city (Toronto) in terms of population size, economic structure, and demographic profile. As illustrated in the following graph.


In [5]:
# Load libraries
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import statsmodels.api as sm

sns.set_style('darkgrid')
import math
from sklearn.linear_model import LogisticRegression
from scipy import stats

import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

from license_model import LicenseModel

### Iteratively run the monthly data for the control group city and get the results coefficients


In [ ]:
import pandas as pd
import os
import numpy as np
from license_model import LicenseModel

# setup the parameters for the license model
BASE_DIR = os.path.join('data', 'QuebecCity')
MONTHS = pd.date_range(start='2024-05-01', end='2025-04-01', freq='MS').strftime('%Y%m').tolist()
MIN_NIGHTS_FOR_LICENSE = 29
LICENSE_REGEX_PATTERN = r'\d{6}'


In [7]:

# Step1 : Process each month's data and collect independent variables
all_variables = set()
protected = []

for month in MONTHS:
    input_file = os.path.join(BASE_DIR, month, 'listings.csv')
    if not os.path.exists(input_file):
        print(f"Missing file: {input_file}")
        continue

    try:
        model = LicenseModel(
            city='Vancouver',
            input_file=input_file,
            min_nights_for_license=MIN_NIGHTS_FOR_LICENSE,
            license_regex_pattern=LICENSE_REGEX_PATTERN,
            protected_columns=protected
        )
        model.run_initial_setup()
        violating_vars = model.check_linearity_of_independent_variables_and_log_odds()
        for var in violating_vars:
            if var in model.listings_df.columns:
                model.listings_df.drop(columns=[var], inplace=True)

        try:
            model.check_no_strongly_influential_outliers(remove_hi_outliers=True)
        except Exception as e:
            print(f"[!] Warning during outlier removal for {month}: {e}")

        model.train_model()

        variables = model.logit_model.result.params.index.tolist()
        all_variables.update([v for v in variables if 'neighbourhood' not in v.lower()])

        print(f"[✓] Processed variables for {month}")

    except Exception as e:
        print(f"[✗] Error processing {month}: {e}")

# Step2 : Create a DataFrame to hold coefficients for each month
Vancouver_airbnb_coefficients = pd.DataFrame(index=sorted(all_variables), columns=MONTHS)

# Step3 : Collect coefficients for each month
for month in MONTHS:
    input_file = os.path.join(BASE_DIR, month, 'listings.csv')
    if not os.path.exists(input_file):
        print(f"Missing file: {input_file}")
        continue

    try:
        model = LicenseModel(
            city='Vancouver',
            input_file=input_file,
            min_nights_for_license=MIN_NIGHTS_FOR_LICENSE,
            license_regex_pattern=LICENSE_REGEX_PATTERN
        )
        model.run_initial_setup()
        violating_vars = model.check_linearity_of_independent_variables_and_log_odds()
        for var in violating_vars:
            if var in model.listings_df.columns:
                model.listings_df.drop(columns=[var], inplace=True)

        try:
            model.check_no_strongly_influential_outliers(remove_hi_outliers=True)
        except Exception as e:
            print(f"[!] Warning during outlier removal for {month}: {e}")

        model.train_model()

        coef_series = model.logit_model.result.params
        coef_series = coef_series[~coef_series.index.str.contains('neighbourhood', case=False, na=False)]

        for var in coef_series.index:
            Vancouver_airbnb_coefficients.loc[var, month] = coef_series[var]

        print(f"[✓] Coefficients recorded for {month}")

    except Exception as e:
        print(f"[✗] Error processing {month}: {e}")

# Step4 : Collect model performance metrics for each month
model_performance_summary = []

for month in MONTHS:
    input_file = os.path.join(BASE_DIR, month, 'listings.csv')
    if not os.path.exists(input_file):
        print(f"Missing file: {input_file}")
        continue

    try:
        model = LicenseModel(
            city='QuebecCity',
            input_file=input_file,
            min_nights_for_license=MIN_NIGHTS_FOR_LICENSE,
            license_regex_pattern=LICENSE_REGEX_PATTERN
        )
        model.run_initial_setup()
        violating_vars = model.check_linearity_of_independent_variables_and_log_odds()
        for var in violating_vars:
            if var in model.listings_df.columns:
                model.listings_df.drop(columns=[var], inplace=True)

        try:
            model.check_no_strongly_influential_outliers(remove_hi_outliers=True)
        except Exception as e:
            print(f"[!] Warning during outlier removal for {month}: {e}")

        model.train_model()
        model.logit_model.evaluate_model(selected_threshold=0.5, print_info=False)

        auc = model.logit_model.auc
        acc = model.logit_model.accuracy
        pseudo_r2 = model.logit_model.result.prsquared
        one_in_ten = model.does_data_follow_one_in_ten_rule(print_info=False)
        n_predictors = model.logit_model.X_train.shape[1]
        illegal_prop = 1 - model.listings_df['legal_listing'].mean()

        model_performance_summary.append({
            'month': month,
            'AUC': auc,
            'Accuracy': acc,
            'Pseudo_R2': pseudo_r2,
            'Num_Predictors': n_predictors,
            'OneInTenRule': one_in_ten,
            'IllegalProp': illegal_prop
        })

        print(f"[✓] Model performance recorded for {month}")

    except Exception as e:
        print(f"[✗] Error processing {month}: {e}")


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 188
         Function evaluations: 253
         Gradient evaluations: 253
[!] Warning during outlier removal for 202405: need covariance of parameters for computing (unnormalized) covariances
Optimization terminated successfully.
         Current function value: 0.000001
         Iterations: 230
         Function evaluations: 302
         Gradient evaluations: 302
                           Logit Regression Results                           
Dep. Variable:          legal_listing   No. Observations:                  904
Model:                          Logit   Df Residuals:                      850
Method:                           MLE   Df Model:                           53
Date:                Tue, 10 Jun 2025   Pseudo R-squ.:                  0.9999
Time:                        01:37:49   Log-Likelihood:            -0.00095771
converged:                       True   LL-Null:          

In [8]:

# Step4 : Save coefficients to CSV
output_path = os.path.join('results', 'Quebec_airbnb_coefficients.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
Vancouver_airbnb_coefficients.to_csv(output_path)

print(f"\n All coefficients saved to: {output_path}")




 All coefficients saved to: results\Quebec_airbnb_coefficients.csv


In [10]:
# Step5 : Save model performance summary to CSV
performance_df = pd.DataFrame(model_performance_summary)
performance_path = os.path.join('results', 'Quebec_airbnb_model_performance.csv')
os.makedirs(os.path.dirname(performance_path), exist_ok=True)
performance_df.to_csv(performance_path, index=False)

print(f"\nAll model performance results saved to: {performance_path}")


All model performance results saved to: results\Quebec_airbnb_model_performance.csv
